<a href="https://colab.research.google.com/github/vamsiporeddy123/CSA6102-DIgital-Forencics/blob/main/Exp_31.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re

AUTH_LINE_RE = re.compile(
    r"(?P<result>Accepted|Failed) password for (?P<user>\S+) from (?P<ip>[\d.]+) port (?P<port>\d+)"
)

def parse_auth_log(lines):
    entries = []

    for line in lines:
        match = AUTH_LINE_RE.search(line)

        if match:
            entries.append({
                "result": match.group("result"),
                "user": match.group("user"),
                "ip": match.group("ip"),
                "port": int(match.group("port")),
                "raw": line
            })

    return entries

def flag_suspicious_logins(entries, trusted_ips):
    flagged = []

    for entry in entries:
        if entry["result"] == "Accepted" and entry["ip"] not in trusted_ips:

            severity = "HIGH" if entry["user"] == "root" else "MEDIUM"

            flagged.append({
                "user": entry["user"],
                "ip": entry["ip"],
                "port": entry["port"],
                "severity": severity,
                "raw": entry["raw"]
            })

    return flagged


log_lines = [
    "Jan 15 08:00:01 server sshd[1001]: Accepted password for deploy from 10.0.0.5 port 51100 ssh2",
    "Jan 15 03:12:01 server sshd[1233]: Failed password for root from 198.51.100.23 port 51320 ssh2",
    "Jan 15 03:12:05 server sshd[1234]: Accepted password for root from 198.51.100.23 port 51322 ssh2"
]

trusted_ips = {"10.0.0.5"}

entries = parse_auth_log(log_lines)

print("=" * 60)
print("SSH AUTHENTICATION LOG ANALYSIS")
print("=" * 60)

for entry in entries:
    print(f"\nUser      : {entry['user']}")
    print(f"Result    : {entry['result']}")
    print(f"IP Address: {entry['ip']}")
    print(f"Port      : {entry['port']}")

flagged = flag_suspicious_logins(entries, trusted_ips)

print("\n" + "=" * 60)
print("SUSPICIOUS LOGIN REPORT")
print("=" * 60)

if flagged:
    for item in flagged:
        print(f"\nUser      : {item['user']}")
        print(f"IP Address: {item['ip']}")
        print(f"Port      : {item['port']}")
        print(f"Severity  : {item['severity']}")
        print("Reason    : Successful login from an untrusted IP.")
else:
    print("No suspicious logins detected.")

print("\n" + "=" * 60)
print("RUNNING TEST CASES")
print("=" * 60)

assert len(entries) == 3
assert len(flagged) == 1
assert flagged[0]["user"] == "root"
assert flagged[0]["ip"] == "198.51.100.23"
assert flagged[0]["severity"] == "HIGH"

print("✓ Test Case 1 Passed: Parsed 3 log entries.")
print("✓ Test Case 2 Passed: Detected 1 suspicious login.")
print("✓ Test Case 3 Passed: Suspicious user is root.")
print("✓ Test Case 4 Passed: Correct IP detected.")
print("✓ Test Case 5 Passed: Severity correctly assigned as HIGH.")

print("\nAll test cases passed successfully.")

SSH AUTHENTICATION LOG ANALYSIS

User      : deploy
Result    : Accepted
IP Address: 10.0.0.5
Port      : 51100

User      : root
Result    : Failed
IP Address: 198.51.100.23
Port      : 51320

User      : root
Result    : Accepted
IP Address: 198.51.100.23
Port      : 51322

SUSPICIOUS LOGIN REPORT

User      : root
IP Address: 198.51.100.23
Port      : 51322
Severity  : HIGH
Reason    : Successful login from an untrusted IP.

RUNNING TEST CASES
✓ Test Case 1 Passed: Parsed 3 log entries.
✓ Test Case 2 Passed: Detected 1 suspicious login.
✓ Test Case 3 Passed: Suspicious user is root.
✓ Test Case 4 Passed: Correct IP detected.
✓ Test Case 5 Passed: Severity correctly assigned as HIGH.

All test cases passed successfully.
